# Firm characteristics: pricing the cross-section instead of forecasting it

[`08a_ipca`](08a_ipca.ipynb) and [`08b_conditional_autoencoder`](08b_conditional_autoencoder.ipynb)
share a two-stage shape: estimate a factor representation, then apply it. The **stochastic
discount factor** model has no such split. It learns one object - a weight on each firm-month that
makes a portfolio of the cross-section price everything else correctly - from a no-arbitrage
condition rather than from a forecasting objective, so there is no factor-return history to hand
to a second stage at all.

The condition it minimizes is that every asset's return, discounted by the SDF, should have zero
expected value. Written out, that is a set of moment conditions, and the model has three moving
parts fitted in phases: an unconditional stage that fixes the SDF weights, a conditional stage
that lets the weights depend on each firm's characteristics and on the state of the market, and a
moment network that searches for the test assets on which the current SDF prices worst - which is
what makes the fit adversarial rather than a plain least-squares problem.

**The forecast is a by-product, and that is the point of reading this notebook against the two
before it.** A firm's predicted return here is what the fitted pricing object implies it should
earn, not something the fit was asked to get right. So a model that prices well and ranks badly
is not a contradiction, and the IC below measures the second while the objective optimized the
first.


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 公司特徵：為橫斷面定價，而非預測橫斷面

[`08a_ipca`](08a_ipca.ipynb) 與 [`08b_conditional_autoencoder`](08b_conditional_autoencoder.ipynb) 共用兩階段結構：先估計因子表示，再加以套用。**隨機折現因子**模型沒有這種切分。它從無套利條件學習一個對象：對每個公司月份的權重，使依橫斷面建構的投資組合能正確為其他所有資產定價，而非以預測為擬合目標。因此完全沒有因子報酬歷史可交給第二階段。

它最小化的條件是：每項資產經 SDF 折現後的報酬，期望值應為零。展開後，這是一組動差條件；模型有三個分階段擬合的部分：固定 SDF 權重的無條件階段、讓權重依各公司特徵及市場狀態而變的條件階段，以及搜尋目前 SDF 定價最差的測試資產之動差網路。最後一部分使擬合成為對抗式程序，而非一般最小平方法問題。

**預測只是副產品，這也是本 Notebook 值得與前兩份對照的重點。** 此處公司的預測報酬，是擬合後定價對象所隱含的應得報酬，而非擬合程序被要求準確預測的數值。因此，模型能有效定價卻不擅長排名並不矛盾；下方資訊係數衡量後者，最佳化目標則是前者。
<!-- ML4T_ZH_END c0000-p000 -->

**Macro context enters here and nowhere else in the family.** `config/setup.yaml` declares eleven
unrevised daily rate and volatility series under `modeling.latent_factors.macro_series`, with a
one-day availability lag. Only unrevised series are eligible: a finalized quarterly figure is not
what anyone could have read at the decision date, and the lag is what stops a same-close
observation reaching a forecast made at that close.

**Learning objectives**

- Separate a pricing objective from a forecasting one, and say what each is optimizing.
- Read a checkpoint surface whose epoch numbers run across training phases rather than within one.
- See why a point-in-time constraint on macro inputs is a data decision rather than a modelling
  preference.

**Book reference**: Chapter 14, Section 14.7 (The stochastic discount factor and the supervised
autoencoder models). Chapter 6, Section 6.7 (Search accounting and run logging) introduces the
run log this notebook writes into.

**Prerequisites**: [`03_financial_features`](03_financial_features.ipynb),
[`04_evaluation`](04_evaluation.ipynb), [`08_latent_factors`](08_latent_factors.ipynb) and
[`08b_conditional_autoencoder`](08b_conditional_autoencoder.ipynb), the two-stage model this one
departs from.

**What it writes**: one training run per label and one complete validation prediction set per
label and epoch checkpoint, in `run_log/registry.db` and under `run_log/training/` and
`run_log/predictions/`, grouped under a population named for this model. The family splits across
four notebooks, so each publishes its own population rather than one shared one.
[`11_backtest`](11_backtest.ipynb) reads them and selects on validation backtest Sharpe.
**Selection happens there, not here**, and the checkpoint is part of what is selected.

<!-- ML4T_ZH_BEGIN c0000-p001 -->
**總體經濟情境只在此處進入此家族。** `config/setup.yaml` 在 `modeling.latent_factors.macro_series` 下宣告十一條未修訂的日度利率與波動率序列，並設有一天的可用時間落後。只有未修訂序列符合資格：最終修訂的季度數字不是決策當時任何人能讀到的數值，而落後設定則防止同一次收盤時才觀察到的數值，進入該次收盤時作出的預測。

**學習目標**

- 區分定價目標與預測目標，並說明兩者各自最佳化的內容。
- 閱讀訓練週期編號跨越不同訓練階段，而非僅在單一階段內累計的檢查點集合。
- 了解對總體經濟輸入施加當時可知資訊限制，是資料決策，而非建模偏好。

**書籍參考**：第 14 章第 14.7 節（隨機折現因子與監督式自編碼器模型）。第 6 章第 6.7 節（搜尋計數與執行紀錄）介紹本 Notebook 寫入的執行紀錄。

**先備條件**：[`03_financial_features`](03_financial_features.ipynb)、[`04_evaluation`](04_evaluation.ipynb)、[`08_latent_factors`](08_latent_factors.ipynb)，以及本模型所偏離的兩階段模型 [`08b_conditional_autoencoder`](08b_conditional_autoencoder.ipynb)。

**寫出內容**：每個標籤各有一次訓練執行；每個標籤與訓練週期檢查點各有一組完整驗證預測，儲存於 `run_log/registry.db`、`run_log/training/` 與 `run_log/predictions/` 之下，並歸入以此模型命名的模型群體。家族分散於四份 Notebook，因此各自發布自己的模型群體，而非共用一個。[`11_backtest`](11_backtest.ipynb) 讀取它們，依驗證回測夏普比率選擇。**選擇在那裡進行，而非此處**，檢查點也是選擇內容的一部分。
<!-- ML4T_ZH_END c0000-p001 -->


In [ ]:
"""Fit the declared firm-characteristics stochastic discount-factor population."""

import plotly.graph_objects as go
import polars as pl
import yaml

from case_studies.research import (
    declared_labels,
    load_model_configs,
    model_requests,
    open_study,
    plan_models,
    planned_model_plan,
    primary_label,
    run_model_population,
    supersedes_for_run,
)
from utils.paths import get_case_study_dir
from utils.style import COLORS, show_plotly_with_alt

In [ ]:
LABELS: list[str] = []
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
PREVIEW_REDUCTIONS: dict = {}
POPULATION_NAME = ""
SUPERSEDES_POPULATION: str = ""
# The device to fit on. Empty means the one declared in config/setup.yaml.
DEVICE: str = ""

MODEL_NAME = "sdf"

In [ ]:
study = open_study(
    "us_firm_characteristics", execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None
)

## 1. Which labels, and what the configuration says

Every label whose training menu declares `latent_factors:` is fitted, and all three do:
`fwd_ret_1m`, `fwd_ret_1m_win` - the same return with each month's cross-section clipped at its
own tails - and `fwd_class_1m`, which turns that return into a class.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 1. 使用哪些標籤，以及設定說明什麼

凡訓練選單宣告 `latent_factors:` 的標籤都會擬合，三種標籤皆然：`fwd_ret_1m`、`fwd_ret_1m_win`——即在各月橫斷面自身的尾端截尾後的相同報酬——以及將該報酬轉成類別的 `fwd_class_1m`。
<!-- ML4T_ZH_END c0004-p000 -->


In [ ]:
declared_labels(study, "latent_factors")

`case_studies/config/sdf/sdf.yaml` declares one epoch budget per phase rather than one for the
model, and the checkpoint surface is built from them. The unconditional stage runs
`n_epochs_unc` epochs and the conditional stage `n_epochs_cond` after it, so a checkpoint's
number is **cumulative across phases**: the first is the end of the unconditional stage, and each
one after it is that plus a point inside the conditional stage. Reading a checkpoint value as an
epoch within one training run would put them in the wrong places relative to each other.

`checkpoint_epochs` lists the points to save at rather than an interval, because the phases are
of different lengths and an interval would land differently in each. `model_kwargs.sdf` in
`config/setup.yaml` narrows that list for this case study.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
`case_studies/config/sdf/sdf.yaml` 為每個階段各宣告訓練週期預算，而非對整個模型只宣告一個預算，檢查點集合由此建立。無條件階段執行 `n_epochs_unc` 個週期，接著條件階段執行 `n_epochs_cond` 個週期，因此檢查點編號**跨階段累計**：第一個是無條件階段結束時，後續各點則是該數值加上條件階段內的某個時點。若將檢查點數值解讀為單一訓練階段內的週期數，就會誤判彼此位置。

`checkpoint_epochs` 逐一列出儲存時點，而非給出固定間隔，因為不同階段長度不同，固定間隔在各階段會落在不同位置。`model_kwargs.sdf` 位於 `config/setup.yaml`，為本案例縮小該清單。
<!-- ML4T_ZH_END c0006-p000 -->


In [ ]:
configs = load_model_configs(
    study,
    "latent_factors",
    labels=LABELS or None,
    config_names=[MODEL_NAME],
)
configs

`LABELS` narrows what is fitted, and a narrowed run declares a different set of members than the
canonical population does. A population is immutable once written, so such a run must publish
under its own name: on a fresh workspace it would otherwise register an incomplete snapshot under
the canonical one, and where the full population already exists the registry refuses it.

The comparison is against this model's own declared rows rather than the whole `latent_factors`
catalog. The family is split across four notebooks and each publishes one model, so the complete
catalog is four times what this one publishes, and comparing against it would report every
canonical run as narrowed.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
`LABELS` 會縮小擬合範圍，而縮小後的執行宣告的成員集合與標準模型群體不同。模型群體一旦寫出便不可變動，因此這類執行必須以專屬名稱發布：否則在全新工作區中，會以標準名稱登記不完整快照；完整群體若已存在，登錄系統則會拒絕。

此處比較的是本模型自身宣告的資料列，而非整個 `latent_factors` 目錄。家族分散於四份 Notebook，各自發布一個模型，因此完整目錄的規模是本 Notebook 發布內容的四倍；若對整個目錄比較，每次標準執行都會被誤報為範圍縮小。
<!-- ML4T_ZH_END c0008-p000 -->


In [ ]:
declared = load_model_configs(study, "latent_factors", config_names=[MODEL_NAME])
if set(configs.get_column("label")) != set(declared.get_column("label")) and not POPULATION_NAME:
    raise ValueError(
        f"this run fits {configs.height} of the {declared.height} declared labels, so it cannot "
        "publish the canonical population; pass POPULATION_NAME to give it its own"
    )

The device is the second thing that can put a run outside the declared population, and it is
not a note beside the result: a network trained on a GPU and the same network trained on a CPU
accumulate their sums in different orders and reach different weights, so the two are different
computations and get different identities. `config/setup.yaml` declares the device this family
publishes on under `modeling.latent_factors`, and the runner refuses a device the machine does
not have rather than falling back to one it does. A reader without an NVIDIA card passes
`DEVICE="cpu"` with a `POPULATION_NAME` of their own.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
裝置是使執行可能落在宣告模型群體之外的第二項因素，而且不只是結果旁的註記：以 GPU 訓練的網路與以 CPU 訓練的相同網路，以不同順序累加總和，最終得到不同權重，因此屬於不同計算，具有不同識別碼。`config/setup.yaml` 在 `modeling.latent_factors` 下宣告此家族發布所用的裝置；若機器沒有要求的裝置，執行器會拒絕，而非自行改用其他裝置。沒有 NVIDIA 顯示卡的讀者，可傳入 `DEVICE="cpu"`，並使用自己的 `POPULATION_NAME`。
<!-- ML4T_ZH_END c0010-p000 -->


In [ ]:
setup = yaml.safe_load(
    (get_case_study_dir("us_firm_characteristics") / "config" / "setup.yaml").read_text()
)
PUBLISHED_DEVICE = str(setup["modeling"]["latent_factors"]["device"])
device = DEVICE or PUBLISHED_DEVICE

if device != PUBLISHED_DEVICE and not POPULATION_NAME:
    raise ValueError(
        f"this run fits on device {device!r} rather than the declared {PUBLISHED_DEVICE!r}, so "
        "it cannot publish the canonical population; pass POPULATION_NAME to give it its own"
    )

## 2. Binding the declarations to the data

Planning reads the label and feature files, loads and lags the declared macro series, computes
the fold boundaries from the walk-forward parameters in `config/setup.yaml`, works out the exact
rows each fit must predict, and derives every training and prediction identity - without fitting
anything. Four things to check:

- **`feature_count` and `eligible_rows` agree across the rows.** A row that differs is a label
  measured on a different sample from its neighbours.
- **`folds` is the same everywhere**, and equals the number of walk-forward splits
  [`04_evaluation`](04_evaluation.ipynb) established.
- **`validation_start` and `validation_end` bracket the development sample**, with none of the
  held-out 2016 tail visible.
- **`checkpoints` is how many training states each label will publish predictions for**, at the
  cumulative epoch numbers described above.

The macro panel does not appear in `feature_count`. It is not a per-firm characteristic: it
enters the conditional stage as the state the SDF weights are allowed to depend on, and its
digest is recorded in the training identity so a different vintage is a different computation.

<!-- ML4T_ZH_BEGIN c0012-p000 -->
## 2. 將宣告與資料連結

規劃過程讀取標籤與特徵檔、載入並延後使用宣告的總體經濟序列、依 `config/setup.yaml` 中的前推參數計算各折邊界、確定每次擬合須預測的精確資料列，並推導所有訓練與預測識別碼；全程不擬合模型。需檢查四件事：

- **`feature_count` 與 `eligible_rows` 在各列一致。** 不同的列表示某標籤是在與其他標籤不同的樣本上衡量。
- **`folds` 在各處相同**，且等於 [`04_evaluation`](04_evaluation.ipynb) 確立的前推切分數。
- **`validation_start` 與 `validation_end` 涵蓋開發樣本首尾**，不顯示任何 2016 年保留測試期間尾段。
- **`checkpoints` 是每個標籤將發布預測的訓練狀態數**，其訓練週期編號依上方所述跨階段累計。

總體經濟縱橫資料不出現在 `feature_count` 中。它不是每家公司的特徵，而是作為條件階段允許 SDF 權重依賴的狀態進入模型；其摘要值會記錄於訓練識別碼，使不同資料版本成為不同的計算。
<!-- ML4T_ZH_END c0012-p000 -->


In [ ]:
requests = model_requests(
    study,
    configs,
    execution_tier=EXECUTION_TIER,
    overrides={"device": device},
    preview_reductions=PREVIEW_REDUCTIONS,
    notebook="08c_stochastic_discount_factor",
)
plan = plan_models(study, requests=requests)

planned_model_plan(plan).select(
    "label",
    "config_name",
    "task",
    "feature_count",
    "eligible_rows",
    "folds",
    "checkpoints",
    "validation_start",
    "validation_end",
)

The plan is also where a short population is visible. A run that fitted fewer labels than the
menu declares, or published fewer checkpoints than the configuration does, would still print an
IC table and still register its rows; what it would not do is announce the gap.

<!-- ML4T_ZH_BEGIN c0014-p000 -->
計畫也是發現模型群體成員不足的地方。若執行擬合的標籤少於選單宣告數，或發布的檢查點少於設定宣告數，仍會列印資訊係數表並登錄資料列，卻不會指出缺口。
<!-- ML4T_ZH_END c0014-p000 -->


In [ ]:
planned_pairs = {(member.label, member.config_name) for member in plan.members}
requested_pairs = set(zip(configs["label"], configs["config_name"], strict=True))
if planned_pairs != requested_pairs:
    raise RuntimeError(
        "the plan does not match the loaded stochastic discount-factor menu; "
        f"missing {sorted(requested_pairs - planned_pairs)}, "
        f"unexpected {sorted(planned_pairs - requested_pairs)}"
    )
print(f"{len(requested_pairs)} label-configuration pairs")
print(f"{len(plan.expected_training_hashes)} training identities")
print(f"{len(plan.expected_prediction_hashes)} validation prediction sets")

## 3. Fitting the population

`run_model_population` runs every planned request. For one request it walks the folds, and on
each one:

1. takes the firm-months inside that fold's training window, and the macro observations dated at
   or before each of those months,
2. fits the unconditional stage: SDF weights that do not vary with the state, trained to make the
   discounted returns price to zero,
3. fits the conditional stage on top, letting the weights depend on each firm's characteristics
   and on the macro state, alternating with a moment network that looks for the portfolios the
   current SDF prices worst,
4. saves the fitted state at each declared checkpoint and, from each one, produces the return
   each validation firm-month is implied to earn.

Step 4 is what makes one fit produce many results. Fold predictions are concatenated into one
series per checkpoint covering the whole validation period, and each becomes its own registered
prediction set with its own identity. Nothing here chooses among them.

**This notebook passes the plan rather than resolved requests, and on this family that changes
what the plan table could show rather than how the fitting is ordered.** The linear, boosted and
TabM adapters each provide a batch runner that prepares one fold set for several configurations;
the latent-factor adapter provides none, so every request is fitted on its own whichever path is
taken. There is nothing here for a batch runner to share in any case: this notebook publishes one
model against three labels, and each label is a different set of rows. What passing the plan does
cost is `eligible_entities`, which needs the eligibility keys themselves; `eligible_rows` above
moves whenever the universe does, so the check that column existed for is still made.


<!-- ML4T_ZH_BEGIN c0016-p000 -->
## 3. 擬合模型群體

`run_model_population` 執行每個已規劃的請求。對單一請求，它逐折處理，每折依序：

1. 取出該折訓練視窗內的公司月份，以及日期不晚於這些月份的總體經濟觀察值；
2. 擬合無條件階段：訓練不隨狀態變動的 SDF 權重，使折現後的報酬定價為零；
3. 在其上擬合條件階段，讓權重依各公司特徵與總體經濟狀態變動，並與尋找當前 SDF 定價最差投資組合的動差網路交替訓練；
4. 在每個宣告的檢查點儲存擬合狀態，並據各狀態產生其隱含的每個驗證公司月份應得報酬。

第 4 步使一次擬合產生多項結果。各折預測依檢查點串接成涵蓋完整驗證期間的序列，每個序列都成為具有自身識別碼的獨立登錄預測集合。此處不在它們之間作選擇。

**本 Notebook 傳入計畫，而非已解析的請求；在此模型家族中，這會改變計畫表可顯示的內容，卻不改變擬合順序。** 線性、提升及 TabM 轉接器各有批次執行器，可為多項設定準備一組折；潛在因子轉接器則沒有，因此無論採哪條路徑，每個請求都單獨擬合。此處也沒有可供批次執行器共用的內容：本 Notebook 對三種標籤發布一個模型，而每種標籤的資料列集合不同。傳入計畫所付出的代價是無法顯示需要資格鍵值本身的 `eligible_entities`；上方的 `eligible_rows` 會隨投資範圍變動，所以該欄原本要進行的檢查仍會執行。
<!-- ML4T_ZH_END c0016-p000 -->

**What the call publishes is a population**: a named, immutable list of the prediction sets it
will produce, written down before the first fit. Afterwards every member must exist and be
complete, which is what makes the downstream comparison well defined.

<!-- ML4T_ZH_BEGIN c0016-p001 -->
**呼叫會發布一個模型群體**：在首次擬合前寫下的具名、不可變動預測集合清單。之後每個成員都必須存在且完整，下游比較才有明確定義。
<!-- ML4T_ZH_END c0016-p001 -->


In [ ]:
population_name = POPULATION_NAME or f"us_firm_characteristics-{MODEL_NAME}-validation-v1"
# The declared hash is only meaningful where a generation of this name already exists. A preview
# run, a reader's first canonical run against an empty `run_log/`, and a run under a caller-chosen
# `POPULATION_NAME` are all refused by `OfficialPopulation.create` if it is passed anyway. The
# resolution lives in shared code so no notebook branches on the tier.
supersedes = supersedes_for_run(
    study,
    population_name=population_name,
    declared=SUPERSEDES_POPULATION,
    execution_tier=EXECUTION_TIER,
)
execution, population = run_model_population(
    study, plan, population_name=population_name, supersedes=supersedes
)

print(f"{len(execution.runs)} configurations, fitted or served from the registry")
print(f"population {population.name}: {len(population.members)} prediction sets")

Re-running this notebook unchanged costs the time it takes to read the data. Every identity is
re-derived from the inputs, the registry already holds the matching rows, and the runner returns
the stored result rather than fitting again.

**There are no fold counts above, and their absence is the honest reading rather than an
omission.** [`05_linear`](05_linear.ipynb) and [`06_gbm`](06_gbm.ipynb) print folds fitted
against folds served from the registry, because the linear and GBM runners record
`fitted_folds` and `reused_folds` on every path they take. The latent-factor runner records
neither - its result carries no diagnostics at all, so the only per-run keys are the status and
the training hash. Printing two zeros here would say every fold came from cache, which is the
opposite of what nothing-recorded means.

[`07_tabular_dl`](07_tabular_dl.ipynb) sits between the two and says so at run time. Its runner
records fold counts when it fits and none when it serves a whole configuration from the
registry, so on a re-run it reports how many of its runs recorded counts instead of printing a
split it was never given.

### Running configurations of your own

The published run log is read-only. To add runs, open the study against a workspace, which holds
its own registry and artifacts and reads the same labels and features:

```python
study = open_study("us_firm_characteristics", workspace="~/ml4t-experiments")
configs = load_model_configs(study, "latent_factors", labels=["fwd_ret_1m"], config_names=["sdf"])
requests = model_requests(study, configs)
plan = plan_models(study, requests=requests)
execution, population = run_model_population(study, plan, population_name="my-sdf-v1")
```


<!-- ML4T_ZH_BEGIN c0018-p000 -->
不修改本 Notebook 直接重跑，只需讀取資料的時間。系統依輸入重新推導各識別碼，登錄系統已有相符資料列，執行器便回傳儲存結果而不重新擬合。

**上方沒有各折計數，這是忠實呈現，而非遺漏。** [`05_linear`](05_linear.ipynb) 與 [`06_gbm`](06_gbm.ipynb) 會印出實際擬合與由登錄系統提供的折數，因為線性及 GBM 執行器在每條路徑都記錄 `fitted_folds` 與 `reused_folds`。潛在因子執行器兩者都不記錄；其結果完全沒有診斷資料，所以每次執行只有狀態與訓練雜湊兩項鍵值。若在此印出兩個零，就會表示每折都來自快取，這與「未記錄」的真意相反。

[`07_tabular_dl`](07_tabular_dl.ipynb) 介於兩者之間，並在執行時說明。其執行器在擬合時記錄各折計數，但由登錄系統提供完整設定時不記錄，因此重跑時會報告有多少次執行記錄了計數，而非印出從未取得的劃分。

### 執行自己的設定

已發布的執行紀錄為唯讀。若要新增執行，請在工作區中開啟研究；工作區有自己的登錄系統與產出檔案，並讀取相同標籤與特徵：

```python
study = open_study("us_firm_characteristics", workspace="~/ml4t-experiments")
configs = load_model_configs(study, "latent_factors", labels=["fwd_ret_1m"], config_names=["sdf"])
requests = model_requests(study, configs)
plan = plan_models(study, requests=requests)
execution, population = run_model_population(study, plan, population_name="my-sdf-v1")
```
<!-- ML4T_ZH_END c0018-p000 -->

The phase budgets and the checkpoint list live in `case_studies/config/sdf/sdf.yaml`, narrowed
for this case study by `model_kwargs.sdf` in `config/setup.yaml`; the macro series and their
availability lag are declared beside it. Each changes this configuration's identity, so the
result registers as a new row beside the old one rather than replacing it.
[`RUN_LOG.md`](../RUN_LOG.md#running-your-own-configurations) covers the rest.

<!-- ML4T_ZH_BEGIN c0018-p001 -->
各階段預算與檢查點清單位於 `case_studies/config/sdf/sdf.yaml`，並由 `model_kwargs.sdf` 在 `config/setup.yaml` 中針對本案例縮小範圍；總體經濟序列及其可用時間落後也在旁邊宣告。任何一項改變都會改變此設定的識別碼，因此新結果會在舊結果旁另登一列，而非取代。其餘細節請參閱 [`RUN_LOG.md`](../RUN_LOG.md#running-your-own-configurations)。
<!-- ML4T_ZH_END c0018-p001 -->


## 4. What came out

One row per label and epoch checkpoint. `ic_mean` is the **information coefficient**: in each
validation month, rank the firms by the model's prediction, rank them by the return they went on
to earn, correlate the two rankings, and average that monthly correlation over the validation
period. It measures ranking, which is not what this model was fitted to do.

**Every count and aggregate below is keyed on `(label, checkpoint_value)`.** Grouping on the
checkpoint alone would average across targets; grouping on the label alone would collapse the
phase structure the checkpoint numbers carry.

The published catalog is checked against the population planned before fitting rather than
against its own row count, because a run that lost a member would otherwise report a shorter
table and nothing else.

<!-- ML4T_ZH_BEGIN c0019-p000 -->
## 4. 產出結果

每個標籤與訓練週期檢查點各占一列。`ic_mean` 是**資訊係數**：在各驗證月份，依模型預測為公司排名，再依其後取得的報酬排名，計算兩組排名的相關性，最後對驗證期間的月度相關性取平均。它衡量排名，而這並非本模型的擬合目標。

**下方所有計數與彙總都以 `(label, checkpoint_value)` 為鍵。** 若只依檢查點分組，會對不同目標取平均；若只依標籤分組，會將檢查點編號所包含的階段結構合併掉。

已發布的目錄會與擬合前規劃的模型群體比較，而非只檢查目錄自身的資料列數；否則少了一個成員的執行只會報告較短的表格，沒有其他提示。
<!-- ML4T_ZH_END c0019-p000 -->


In [ ]:
catalog = execution.catalog_rows.select(
    "config_name",
    "label",
    "task",
    "complete",
    "checkpoint_kind",
    "checkpoint_value",
    "ic_mean",
    "ic_std",
    "ic_t",
    "ic_n_days",
    "auc_mean_daily",
    pl.col("direction_label").alias("auc_scored_against"),
    "n_folds",
    "training_hash",
    "prediction_hash",
).sort(["label", "checkpoint_value"])

if set(catalog.get_column("prediction_hash")) != set(plan.expected_prediction_hashes):
    raise RuntimeError("the published catalog differs from the population planned before fitting")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("a partial stochastic discount-factor checkpoint cannot pass to backtesting")
if catalog.select("label", "config_name", "checkpoint_value").n_unique() != catalog.height:
    raise RuntimeError("each label and checkpoint must identify one prediction set")
if catalog.get_column("checkpoint_value").null_count():
    raise RuntimeError("every checkpoint must name the cumulative epoch it was taken at")

catalog = catalog.with_columns(
    full_coverage=pl.col("ic_n_days") == pl.col("ic_n_days").max().over("label")
)

primary = primary_label(study)
present = sorted(set(catalog.get_column("label")))
# The primary label leads when it was fitted. A subset run that leaves it out orders by whichever
# label it did fit rather than by one that is not there.
panel_labels = [label for label in [primary] if label in present] + [
    label for label in present if label != primary
]
print(f"{catalog.height} candidate models across {len(panel_labels)} labels")
print(f"at {catalog.n_unique('checkpoint_value')} checkpoints each")
catalog.select(
    "label",
    "checkpoint_value",
    "ic_mean",
    "ic_std",
    "ic_t",
    "ic_n_days",
    "full_coverage",
).head(15)

### The ranking across the two training phases

Each line traces one label's out-of-sample IC against the cumulative epoch count, so the leftmost
point is the end of the unconditional stage and everything to its right is the conditional stage
refining it. That boundary is the one place on this chart where the model changes shape rather
than merely trains longer, and it is marked.

What the chart cannot say is whether the fit is pricing better, because the vertical axis is a
ranking measure and the objective is a pricing one. A line that falls after the phase boundary is
a model whose ranking got worse while it was being asked to price better, which is a real thing
for this family rather than a failure to converge.

<!-- ML4T_ZH_BEGIN c0021-p000 -->
### 跨兩個訓練階段的排名

每條線追蹤一種標籤的樣本外資訊係數與累積訓練週期數的關係，因此最左端是無條件階段結束，右側全是條件階段的進一步調整。這條邊界是圖中唯一一個模型形狀改變、而非僅增加訓練時間的位置，圖上已予標示。

圖表無法說明定價是否改善，因為縱軸衡量排名，目標函數則衡量定價。若某條線在階段邊界後下降，表示模型在被要求改善定價的同時，排名能力變差；對此家族而言，這是確實可能發生的現象，而非未收斂。
<!-- ML4T_ZH_END c0021-p000 -->


In [ ]:
curves = catalog.filter("full_coverage").sort("label", "checkpoint_value")
phase_boundary = int(curves.get_column("checkpoint_value").min())
fig = go.Figure()
for label in panel_labels:
    series = curves.filter(pl.col("label") == label).sort("checkpoint_value")
    fig.add_trace(
        go.Scatter(
            x=series.get_column("checkpoint_value").to_list(),
            y=series.get_column("ic_mean").to_list(),
            mode="lines+markers",
            name=f"{label} ({'primary' if label == primary else 'variant'})",
            line=dict(
                color=COLORS["blue"] if label == primary else COLORS["slate"],
                width=2 if label == primary else 1.5,
            ),
            marker=dict(size=6),
        )
    )
fig.add_hline(y=0, line_width=1, line_dash="dash", line_color=COLORS["neutral"])
fig.add_vline(
    x=phase_boundary,
    line_width=1,
    line_dash="dot",
    line_color=COLORS["copper"],
    annotation_text="conditional stage begins",
    annotation_position="top right",
)
fig.update_xaxes(title_text="Cumulative training epochs across both phases")
fig.update_yaxes(title_text="Mean IC (validation)")
fig.update_layout(
    title="A pricing objective is not asked to rank, and the path shows it",
    height=460,
    width=900,
    margin=dict(t=90),
    legend=dict(title_text="Label"),
)
# Which direction each line moves after the phase boundary is a fact about the frame, so the alt
# text reads it rather than asserting a shape the next run may not reproduce.
moves = (
    curves.group_by("label")
    .agg(
        at_boundary=pl.col("ic_mean").sort_by("checkpoint_value").first(),
        at_end=pl.col("ic_mean").sort_by("checkpoint_value").last(),
    )
    .sort("label")
)
move_text = "; ".join(
    f"{row['label']} {'rises' if row['at_end'] > row['at_boundary'] else 'falls'} across the "
    "conditional stage"
    for row in moves.iter_rows(named=True)
)
show_plotly_with_alt(
    fig,
    "Line chart of mean validation information coefficient against cumulative training epochs, one "
    "line per label with a marker at each declared checkpoint: the primary target in dark navy and "
    "the two variants in slate, over a dashed zero line, with a dotted copper vertical line where "
    f"the conditional stage begins. Counted from the frame: {move_text}.",
)

### How much the stopping point is worth

The frame below puts the range a label's IC covers across its own checkpoints against the spread
across labels at the final one. That is the quantity deciding whether the stopping point is a
decision worth making carefully or one being made by noise, and both sides are computed within
this model so the comparison is not smuggling in a difference between models.

<!-- ML4T_ZH_BEGIN c0023-p000 -->
### 停止時點有多少價值

下方資料框將某標籤自身檢查點之間的資訊係數範圍，與最後一個檢查點時各標籤之間的差距相比。這個量決定停止時點是值得謹慎選擇的決策，還是主要受雜訊影響；兩側都在此模型內計算，所以比較不會悄悄混入模型間的差異。
<!-- ML4T_ZH_END c0023-p000 -->


In [ ]:
final_epoch = int(curves.get_column("checkpoint_value").max())
checkpoint_vs_label = (
    curves.group_by("label")
    .agg(
        ic_min=pl.col("ic_mean").min(),
        ic_max=pl.col("ic_mean").max(),
        ic_at_boundary=pl.col("ic_mean")
        .filter(pl.col("checkpoint_value") == phase_boundary)
        .first(),
        ic_final=pl.col("ic_mean").filter(pl.col("checkpoint_value") == final_epoch).first(),
        scored_months=pl.col("ic_n_days").max(),
    )
    .with_columns(checkpoint_range=pl.col("ic_max") - pl.col("ic_min"))
    .sort("label")
)
across_labels = float(
    checkpoint_vs_label.get_column("ic_final").max()
    - checkpoint_vs_label.get_column("ic_final").min()
)
print(f"unconditional stage ends at epoch {phase_boundary}; last checkpoint is {final_epoch}")
print(f"spread across labels at the last checkpoint: {across_labels:.4f}")
checkpoint_vs_label

## 5. What to notice

**The objective here is not the metric here, and that gap is the lesson rather than a defect.**
The fit minimizes pricing error under a no-arbitrage condition; `ic_mean` measures whether the
implied returns rank the cross-section. Those can move together and they need not. Reading a low
IC as the model failing would be reading it against a target it was never given - and reading a
high one as vindication of the pricing would be the same error in the other direction.

**Where this sits relative to the two notebooks before it.**
[`08a_ipca`](08a_ipca.ipynb) and [`08b_conditional_autoencoder`](08b_conditional_autoencoder.ipynb)
both estimate a factor representation and then apply it, and differ only in whether the map from
characteristics to exposures is linear. This model has no such two-stage split to make: the
pricing object is what is learned, and the forecast falls out of it. So the three of them are not
three points on one axis, and the family's four members are better read as two pairs.

**The macro constraint is a data decision and it costs coverage.** Only unrevised daily series
are eligible, which rules out most published macro aggregates, and the one-day availability lag
rules out the same-close reading. Both are what make the conditional stage's state variable
something a reader could actually have observed at the decision date. A version of this model
conditioned on revised quarterly data would look better and would not be a forecast.

**None of this selects anything.** IC measures whether predictions rank firms correctly, not
whether a strategy trading them makes money after costs and turnover. Selection is on validation
backtest Sharpe in [`11_backtest`](11_backtest.ipynb), where the checkpoint is part of what is
selected.


<!-- ML4T_ZH_BEGIN c0025-p000 -->
## 5. 值得注意的結果

**此處的最佳化目標不是評分指標；兩者之間的差距是本節要說明的重點，而非缺陷。** 擬合過程在無套利條件下最小化定價誤差；`ic_mean` 衡量隱含報酬能否排序橫斷面。兩者可能同向變動，也可能不會。若將低資訊係數解讀為模型失敗，就是以模型從未接收的目標衡量它；若將高資訊係數解讀為定價能力已獲證實，則是相反方向的相同錯誤。

**此模型相對於前兩份 Notebook 的位置。** [`08a_ipca`](08a_ipca.ipynb) 與 [`08b_conditional_autoencoder`](08b_conditional_autoencoder.ipynb) 都先估計因子表示、再加以套用，只在特徵到曝險的映射是否為線性函數上不同。本模型沒有這種兩階段切分：學得的是定價對象，預測則是其衍生結果。因此三者不是同一座標軸上的三個點，家族的四個成員更適合視為兩組配對。

**總體經濟資料限制是一項資料決策，也會減少涵蓋範圍。** 只有未修訂的日度序列符合資格，這排除了多數已發布的總體經濟彙總數據；一天的可用時間落後則排除了同次收盤才取得的讀數。兩者共同使條件階段的狀態變數，成為讀者在決策日期確實可能觀察到的資訊。若模型使用事後修訂的季度資料，結果可能看起來更好，卻不能算預測。

**上述結果都不作出選擇。** 資訊係數衡量預測是否正確排序公司，而非交易策略扣除成本及週轉後能否獲利。選擇依據是 [`11_backtest`](11_backtest.ipynb) 中的驗證回測夏普比率，檢查點也是選擇內容的一部分。
<!-- ML4T_ZH_END c0025-p000 -->

**Known limitations.** The phase budgets, the checkpoint list, the factor count and the network
widths are declared rather than chosen, and nothing here searches over them. The adversarial
moment network makes the fit sensitive to its own optimization path in a way a least-squares
problem is not, so two runs at different thread counts are not guaranteed to agree to full
precision even at a fixed seed - the runtime is inside the training identity for that reason. The
IC is an average of monthly rank correlations with no adjustment for serial dependence, so `ic_t`
is a diagnostic rather than a test. And every number is measured on validation folds that have
been read many times over by the time a case study reaches this notebook.

<!-- ML4T_ZH_BEGIN c0025-p001 -->
**已知限制。** 各階段預算、檢查點清單、因子數及網路寬度都是宣告值，而非經選擇所得；此處不對它們搜尋。對抗式動差網路使擬合結果對自身最佳化路徑敏感，這與最小平方法問題不同；因此即使種子固定，執行緒數不同的兩次執行也不保證達到完全相同的精度，這也是執行環境納入訓練識別碼的原因。資訊係數是月度排名相關性的平均值，未調整序列相依性，因此 `ic_t` 是診斷，而非檢驗。此處每個數字都在驗證折上衡量，而案例研究進行至本 Notebook 時，這些驗證折已被反覆查看多次。
<!-- ML4T_ZH_END c0025-p001 -->


**Next**: [`08d_supervised_autoencoder`](08d_supervised_autoencoder.ipynb) is the other member
that breaks the two-stage shape, and it breaks it from the opposite end: no pricing condition and
no factor intermediate, just characteristics mapped to forward returns directly.

<!-- ML4T_ZH_BEGIN c0026-p000 -->
**接下來**：[`08d_supervised_autoencoder`](08d_supervised_autoencoder.ipynb) 是另一個打破兩階段結構的成員，而且從相反方向打破：沒有定價條件，也沒有中途因子表示，只將特徵直接映射至前向報酬。
<!-- ML4T_ZH_END c0026-p000 -->
